# 03 — Batch URL triage

Before investigating utilities one by one, run this once. It probes every
utility URL in the starter list, flags dead URLs, and pre-classifies each as
KUBRA / ArcGIS / unknown — so you know exactly where the real work is instead
of investigating blind.

This does **not** register anything and does **not** replace the DevTools step
in notebook 01. It produces a prioritized worklist.

## Setup

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "config.py").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
print("repo root:", root)

import pandas as pd

## Step 1 — Run the batch check

This sends one polite HTTP request per utility, so ~100 utilities takes a few
minutes. Use `--limit` while testing. Results are written to
`starter/url_triage.csv`.

In [ ]:
# Run the batch checker as a subprocess so its polite per-host delays apply.
import subprocess
result = subprocess.run(
    [sys.executable, "batch_check.py"],      # add "--limit", "20" to test fast
    cwd=str(root), capture_output=True, text=True,
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])

## Step 2 — Read the worklist

In [ ]:
triage = pd.read_csv(root / "starter" / "url_triage.csv")
print(f"{len(triage)} utilities triaged\n")

print("URL status:")
print(f"  alive : {int(triage['url_alive'].sum())}")
print(f"  dead  : {int((~triage['url_alive']).sum())}")
print("\nPlatform guesses (alive URLs only):")
print(triage[triage['url_alive']]['platform_guess'].value_counts().to_string())

## Step 3 — The three work buckets

Each utility falls into one of three buckets. Work them in this order.

In [ ]:
alive = triage[triage['url_alive']]

print("=== BUCKET 1: KUBRA — batch these, same DevTools moves each ===")
kubra = alive[alive['platform_guess'] == 'kubra']
print(kubra[['rank', 'utility_id', 'utility_name', 'states']].to_string(index=False))

print("\n=== BUCKET 2: ArcGIS — batch these together ===")
arc = alive[alive['platform_guess'] == 'arcgis']
print(arc[['rank', 'utility_id', 'utility_name', 'states']].to_string(index=False))

print("\n=== BUCKET 3: unknown — investigate individually (may be new platforms) ===")
unk = alive[alive['platform_guess'] == 'unknown']
print(unk[['rank', 'utility_id', 'utility_name', 'states']].to_string(index=False))

## Step 4 — Dead URLs to fix

In [ ]:
dead = triage[~triage['url_alive']]
if dead.empty:
    print("No dead URLs - every starter URL responded.")
else:
    print(f"{len(dead)} URL(s) need a fresh link. Search "
          f"'<utility> outage map' for each:\n")
    print(dead[['rank', 'utility_id', 'utility_name', 'url', 'note']].to_string(index=False))

---

You now have a prioritized worklist. Take it into `01_build_registry.ipynb`:
work the KUBRA bucket first (fastest — repeat the same moves), then ArcGIS,
then investigate the unknowns. Fix dead URLs as you reach them. You do **not**
need all 100+ before collecting — verify 10–20, schedule the collector, and
keep adding.